In [15]:
import pandas as pd

df_combined=pd.read_csv("../data/elexon_full_prices.csv")

# ensure types
df_combined["settlementDate"] = pd.to_datetime(df_combined["settlementDate"], errors="coerce")
df_combined["netImbalanceVolume"] = pd.to_numeric(df_combined["netImbalanceVolume"], errors="coerce")
df_combined["systemSellPrice"] = pd.to_numeric(df_combined["systemSellPrice"], errors="coerce")
df_combined["systemBuyPrice"]  = pd.to_numeric(df_combined["systemBuyPrice"],  errors="coerce")

# daily midpoint price
df_daily = (
    df_combined.groupby("settlementDate")[["systemSellPrice","systemBuyPrice"]]
      .mean()
      .rename(columns={"systemSellPrice":"ssp","systemBuyPrice":"sbp"})
      .assign(price=lambda x: x[["ssp","sbp"]].mean(axis=1))
      .reset_index()
      .sort_values("settlementDate")
)

# NIV daily (sum half-hours → daily)
niv_daily = (
    df_combined.groupby("settlementDate")["netImbalanceVolume"]
      .sum()
      .reset_index()
      .rename(columns={"netImbalanceVolume":"NIV"})
)

# merge
df_daily = df_daily.merge(niv_daily, on="settlementDate", how="left")
df_daily.head(100)


,settlementDate,ssp,sbp,price,NIV
0,2025-02-01,111.909498,111.909498,111.909498,3564.192452
1,2025-02-02,96.160760,96.160760,96.160760,-4958.656688
2,2025-02-03,102.267979,102.267979,102.267979,-13457.367424
3,2025-02-04,119.117084,119.117084,119.117084,8252.535702
4,2025-02-05,126.262515,126.262515,126.262515,7094.496854
...,...,...,...,...,...
95,2025-05-07,91.194811,91.194811,91.194811,3601.908344
96,2025-05-08,101.995970,101.995970,101.995970,4854.528804
97,2025-05-09,107.630892,107.630892,107.630892,4997.203308
98,2025-05-10,73.152373,73.152373,73.152373,1658.593938


In [16]:
df_daily["settlementDate"].value_counts().head()

settlementDate
2025-02-01    1
2025-02-02    1
2025-02-03    1
2025-02-04    1
2025-02-05    1
Name: count, dtype: int64

In [17]:
df_daily[["price","NIV"]].corr()
df_daily.isna().sum()

settlementDate    0
ssp               0
sbp               0
price             0
NIV               0
dtype: int64

In [18]:
df_daily = df_daily.sort_values("settlementDate").reset_index(drop=True)
df_daily["NIV_lag1"] = df_daily["NIV"].shift(1)
df_daily["NIV_roll7"] = df_daily["NIV"].rolling(7, min_periods=3).mean()

In [22]:
from prophet import Prophet
from sklearn.metrics import mean_absolute_percentage_error as mape

df_p = df_daily.rename(columns={"settlementDate":"ds","price":"y"}).dropna(subset=["ds","y","NIV"])

split = int(len(df_p)*0.85)
train, test = df_p.iloc[:split].copy(), df_p.iloc[split:].copy()

m = Prophet(weekly_seasonality=True, daily_seasonality=False)
m.add_regressor("NIV")
m.fit(train)

pred = m.predict(test[["ds","NIV"]])[["ds","yhat"]].set_index("ds")
score = mape(test.set_index("ds")["y"], pred["yhat"])
print(f"MAPE with NIV: {score:.2%}")


17:50:51 - cmdstanpy - INFO - Chain [1] start processing
17:50:51 - cmdstanpy - INFO - Chain [1] done processing


MAPE with NIV: 55.65%


In [21]:
# Saving elexon daily with net imbalance volume data
df_daily.to_csv("../data/elexon_daily_with_niv.csv", index=False)